### `etd_rk_core.ipynb`
*Created: September 26, 2026* <br/>
This notebook implements the core framework for the ETD-RK solvers. 

In [ ]:
struct SemilinearODEProblem{M, F, U, P, S}
    A::M
    f::F
    u0::U
    tspan::Tuple{Float64, Float64}
    p::P
    exact_solution::S
end 

function SemilinearODEProblem(A, f, u0, tspan, p; exact_solution = nothing)
    return SemilinearODEProblem(A, f, u0, tspan, p, nothing)
end 

mutable struct ETDRK43Cache{M, V}

    #Linear term (scaled)
    M::M  #hA
    N::M  #(h/2)A 

    #Exponential matrices (or scalars if hA is a scalar)
    E::M   #ϕ₀(hA)
    G::M   #ϕ₀(hA/2)

    #Phi matrices (full step)
    P::Vector{M}   #[ϕ₁(hA), ϕ₂(hA), ϕ₃(hA)]

    #Phi matrices (half step)
    Q::Vector{M}   #[ϕ₁(hA/2), ϕ₂(hA/2), ϕ₃(hA/2)]

    #Stages 
    K::Vector{V}   #[K₁, K₂, K₃, K₄, K₅]   
    
    #Nonlinear evals 
    F::Vector{V}   #[F₁, F₂, F₃, F₄, F₅]   

    #Weights
    B::Vector{M}   #[B₁, B₂, B₃, B₄] 

    #Candiate for u_{n+1}
    u_candidate::V
    
    #Error estimate e_n 
    error_estimate::V
end 

In [ ]:
#Outer constructor for cache 
function ETDRK43Cache(u0::Number)

    return ETDRK43Cache(0.0,        #M
                        0.0,        #N
                        0.0,        #E
                        0.0,        #G
                        zeros(3),   #P
                        zeros(3),   #Q
                        zeros(5),   #K
                        zeros(5),   #F
                        zeros(4),   #B
                        0.0,        #u_candidate
                        0.0         #error_estimate
                        )
end 

In [ ]:
 function ETDRK43Cache(u0::Vector{Float64})
    """
    WARNING: Do not use the same instance to initialize multiple buffers!
             If you do this then mutating M, N, E or G will update all fields that were constructed from that instance!
    """    

    vector_buffer() = zeros(length(u0))
    matrix_buffer() = zeros(length(u0), length(u0))
    
    return ETDRK43Cache(matrix_buffer(),                  #M
                        matrix_buffer(),                  #N
                        matrix_buffer(),                  #E
                        matrix_buffer(),                  #G
                        [matrix_buffer() for _ in 1:3],   #P
                        [matrix_buffer() for _ in 1:3],   #Q
                        [vector_buffer() for _ in 1:5],   #K
                        [vector_buffer() for _ in 1:5],   #F
                        [matrix_buffer() for _ in 1:4],   #B
                        vector_buffer(),                  #u_candidate (vector)
                        vector_buffer()                   #error_estimate (vector)
                    )
end 